# Churn Modeling 07-3 - AUC & F1 Optimization

이번 노트북의 목표는 retention action 해석이 아니라 **순수 예측 성능 개선**이다.

우선순위는 다음 두 가지다.

1. **AUC를 높인다.** 여기서는 기본적으로 ROC AUC를 의미하고, 보조로 PR AUC도 함께 본다.
2. **F1 score를 높인다.** F1은 threshold에 민감하므로 valid set에서 threshold를 튜닝한 뒤 test set에서 최종 확인한다.

`07-2`에서 많은 behavior feature가 top-k targeting에는 도움이 크지 않았으므로, `07-3`에서는 원본 피처를 강한 baseline으로 두고, 소수의 compact feature만 비교한다.


## 피쳐 설명

이 노트북은 AUC와 F1 개선을 목적으로 `07-1`, `07-2`보다 보수적인 compact feature set만 사용한다. 많은 파생변수를 넣기보다, 성능 개선 가능성이 높은 숫자형 행동 신호와 소수의 위험 flag를 중심으로 비교한다.

| 구분 | 피쳐명 | 설명 |
| --- | --- | --- |
| 사용 강도 | `watch_time_per_session` | 세션당 평균 시청 시간 |
| 계정 연령 대비 사용 | `watch_time_per_account_age` | 계정 연령 대비 주당 평균 시청 시간 |
| 계정 연령 대비 사용 | `session_per_account_age` | 계정 연령 대비 접속 세션 수 |
| 완료 행동 | `completion_per_watch_time` | 시청 시간 대비 콘텐츠 완료율 |
| 추천 반응 | `recommendation_completion_gap` | 추천 클릭률과 완료율의 차이 |
| 비활성 | `inactive_ratio_to_account_age` | 계정 연령 대비 미접속 기간 비율 |
| Engagement | `engagement_score` | 시청 시간, 시청 세션, 완료율, 추천 반응, 최근 접속성을 p95 기준으로 정규화해 결합한 점수 |
| 사용량 flag | `low_watch_flag` | 시청 시간이 train 1사분위수 이하인 사용자 flag |
| 사용량 flag | `low_session_flag` | 시청 세션 수가 train 1사분위수 이하인 사용자 flag |
| 완료 flag | `low_completion_flag` | 완료율이 train 1사분위수 이하인 사용자 flag |
| 추천 flag | `low_reco_flag` | 추천 클릭률이 train 1사분위수 이하인 사용자 flag |
| 비활성 flag | `high_inactivity_flag` | 미접속 기간이 train 3사분위수 이상인 사용자 flag |
| 결합 위험 | `inactive_low_watch_flag` | 높은 비활성과 낮은 시청량이 동시에 나타나는 사용자 flag |
| 결합 위험 | `low_interest_flag` | 낮은 추천 반응과 낮은 완료율이 동시에 나타나는 사용자 flag |
| 범주 조합 | `basic_mobile_flag` | Basic 요금제와 Mobile 중심 사용 조합 |
| 위험 집계 | `high_risk_behavior_count` | compact 위험 flag들의 합계 |

주의: `07-3`에서는 feature engineering 자체보다 valid set threshold tuning과 AUC/F1 기준 후보 비교가 핵심이다.

## 1. 라이브러리 로드

AUC/F1 개선에 집중하기 위해 Logistic Regression, LightGBM, XGBoost, CatBoost를 중심으로 비교한다. 설치되어 있지 않은 모델은 자동으로 skip한다.


In [1]:
from pathlib import Path
import warnings

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import randint, loguniform, uniform

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
N_SPLITS = 3
N_ITER = 15
N_JOBS = -1

font_candidates = ['AppleGothic', 'NanumGothic', 'Malgun Gothic']
available_fonts = {font.name for font in fm.fontManager.ttflist}
font_name = next((font for font in font_candidates if font in available_fonts), None)
if font_name:
    plt.rcParams['font.family'] = font_name
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')


def make_ohe(**kwargs):
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False, **kwargs)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False, **kwargs)


## 2. 데이터 로드와 Train / Valid / Test Split

F1을 높이려면 threshold tuning이 필요하다. 따라서 test set에서 threshold를 고르면 안 된다.

데이터가 5만 행이므로 valid/test를 각각 10%로 둬도 약 5천 건씩 확보된다. AUC/F1 최적화 목적에서는 학습 데이터를 더 많이 확보하는 `80/10/10` split이 더 유리하다.

- Train 80%: 모델 학습과 hyperparameter search에 사용
- Valid 10%: threshold 선택과 최종 후보 선택에 사용
- Test 10%: 마지막 성능 확인에만 사용


In [2]:
data_candidates = [
    Path('data/user_behavior_50000/netflix_user_behavior_churn_50000v2.csv'),
    Path('../data/user_behavior_50000/netflix_user_behavior_churn_50000v2.csv'),
]
data_path = next(path for path in data_candidates if path.exists())

df = pd.read_csv(data_path)
target_col = 'churned' if 'churned' in df.columns else 'churn'

X = df.drop(columns=['user_id', target_col])
y = df[target_col]

# 1차 split: test 10% 고정
X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X,
    y,
    test_size=0.10,
    stratify=y,
    random_state=RANDOM_STATE,
)

# 2차 split: train 80%, valid 10%, test 10%
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid,
    y_train_valid,
    test_size=1/9,
    stratify=y_train_valid,
    random_state=RANDOM_STATE,
)

print('Data path:', data_path)
print('Target column:', target_col)
print('Train shape:', X_train.shape, 'Valid shape:', X_valid.shape, 'Test shape:', X_test.shape)
print('Train churn ratio:', round(y_train.mean(), 4))
print('Valid churn ratio:', round(y_valid.mean(), 4))
print('Test churn ratio:', round(y_test.mean(), 4))
X_train.head()


Data path: ../data/user_behavior_50000/netflix_user_behavior_churn_50000v2.csv
Target column: churned
Train shape: (40000, 18) Valid shape: (5000, 18) Test shape: (5000, 18)
Train churn ratio: 0.2093
Valid churn ratio: 0.2092
Test churn ratio: 0.2092


,age,gender,region,subscription_type,payment_method,primary_device,account_age_months,favorite_genre,time_of_day,recommendation_source,session_count,avg_watch_time_minutes_per_week,watch_sessions_per_week,completion_rate,avg_rating_given,app_rating,recommendation_click_rate,days_since_last_login
33871,21,Male,Africa,Basic,Debit Card,Smart TV,7,Drama,Evening,Trending,1,120,2,72,5,4,12,2
36832,30,Male,South America,Basic,Debit Card,Mobile,40,Documentary,Afternoon,Algorithm,1,141,4,74,3,3,54,33
22002,19,Male,North America,Basic,Paypal,Smart TV,54,Action,Evening,Friend,1,266,7,89,3,4,41,6
11820,43,Female,South America,Standard,Credit Card,Mobile,74,Action,Evening,Friend,3,142,5,66,3,4,51,51
30475,20,Female,Europe,Basic,Credit Card,Smart TV,35,Drama,Night,Search,1,820,19,82,5,5,35,5


## 3. Compact Feature Engineering

`07-2`에서는 많은 feature가 오히려 성능을 떨어뜨렸다. 여기서는 AUC/F1 개선 가능성이 있는 소수 피처만 만든다.

방향은 다음과 같다.

- 원본 컬럼은 유지한다.
- 과도한 categorical interaction은 만들지 않는다.
- ratio, recency, engagement, risk count처럼 모델이 바로 활용하기 쉬운 숫자형 신호 위주로 만든다.
- quantile 기준은 train에서만 fit한다.


## 3-1. 07-1 / 07-2 대비 수정 방향

`07-3`의 feature engineering은 `07-1`, `07-2`보다 훨씬 보수적으로 구성한다. 이번 목표가 캠페인 해석이 아니라 **AUC와 F1 score 개선**이기 때문이다.

**07-1 대비 수정점**

`07-1`은 First Try 성격의 실험이었다. ratio feature, risk flag, categorical interaction, bin feature를 넓게 추가했다. 예를 들어 subscription/device 조합, device/time 조합, inactive bin, watch time bin처럼 다양한 파생변수를 만들었다.

하지만 피처 수가 늘어난다고 성능이 자동으로 좋아지지는 않았다. 모델 입장에서는 유용한 신호와 noise가 함께 들어갔고, 실제 성능 개선도 뚜렷하지 않았다. 또한 단순 함수형 feature engineering이라 quantile 기준을 train에서 fit하고 test에 transform하는 구조가 아니었다.

`07-3`에서는 이 문제를 줄이기 위해 다음처럼 수정한다.

- categorical interaction을 거의 제거한다.
- bin feature를 제거한다.
- 너무 많은 risk flag를 제거한다.
- `BaseEstimator`, `TransformerMixin` 기반 transformer 구조를 유지한다.
- quantile 기준은 train 데이터에서만 fit한다.
- AUC/F1에 도움 될 가능성이 높은 숫자형 행동 피처만 남긴다.

**07-2 대비 수정점**

`07-2`는 leakage-safe 구조는 좋았지만, retention campaign action matrix까지 고려했기 때문에 파생변수가 많았다. 총 35개 feature가 추가됐고, `retention_action_segment`, 여러 combo feature, 다양한 risk flag가 포함됐다.

그 결과 해석 가능성은 좋아졌지만, 실제 모델 성능에서는 PR AUC, ROC AUC, top 10% lift가 대부분 하락했다. 즉 `07-2`의 feature engineering은 상위 risk 고객을 설명하고 캠페인 액션을 배정하는 데는 유용하지만, 예측 모델의 기본 입력으로 쓰기에는 과했다.

`07-3`에서는 이 결론을 반영해 compact feature만 사용한다. 최종 방향은 다음과 같다.

> 피처 수를 늘리는 것이 아니라, 모델이 잘 학습할 가능성이 높은 행동 신호만 남기고, 성능 개선은 compact feature, hyperparameter tuning, valid threshold tuning으로 가져간다.


In [3]:
class CompactChurnFeatureEngineer(BaseEstimator, TransformerMixin):
    """AUC/F1 개선을 목적으로 한 compact feature transformer.

    07-2의 많은 피처 중 해석성과 예측 가능성이 높은 소수 피처만 사용한다.
    quantile cutoff는 fit 단계에서 train 데이터 기준으로만 저장한다.
    """

    def __init__(self, eps=1e-6):
        self.eps = eps

    def fit(self, X, y=None):
        X_fit = X.copy()
        self.watch_p95_ = max(X_fit['avg_watch_time_minutes_per_week'].quantile(0.95), self.eps)
        self.session_p95_ = max(X_fit['watch_sessions_per_week'].quantile(0.95), self.eps)
        self.inactive_p95_ = max(X_fit['days_since_last_login'].quantile(0.95), self.eps)

        self.watch_low_q_ = X_fit['avg_watch_time_minutes_per_week'].quantile(0.25)
        self.session_low_q_ = X_fit['watch_sessions_per_week'].quantile(0.25)
        self.completion_low_q_ = X_fit['completion_rate'].quantile(0.25)
        self.reco_low_q_ = X_fit['recommendation_click_rate'].quantile(0.25)
        self.inactive_high_q_ = X_fit['days_since_last_login'].quantile(0.75)
        return self

    def transform(self, X):
        X_fe = X.copy()
        eps = self.eps

        # Ratio features: 사용 강도와 세션당 몰입도.
        X_fe['watch_time_per_session'] = (
            X_fe['avg_watch_time_minutes_per_week']
            / (X_fe['watch_sessions_per_week'] + eps)
        )
        X_fe['watch_time_per_account_age'] = (
            X_fe['avg_watch_time_minutes_per_week']
            / (X_fe['account_age_months'] + 1)
        )
        X_fe['session_per_account_age'] = (
            X_fe['session_count']
            / (X_fe['account_age_months'] + 1)
        )
        X_fe['completion_per_watch_time'] = (
            X_fe['completion_rate']
            / (X_fe['avg_watch_time_minutes_per_week'] + 1)
        )
        X_fe['recommendation_completion_gap'] = (
            X_fe['recommendation_click_rate'] - X_fe['completion_rate']
        )
        X_fe['inactive_ratio_to_account_age'] = (
            X_fe['days_since_last_login']
            / (X_fe['account_age_months'] * 30 + 1)
        )

        # Engagement score: 0~100 스케일의 compact signal.
        watch_norm = np.clip(X_fe['avg_watch_time_minutes_per_week'] / self.watch_p95_, 0, 1)
        session_norm = np.clip(X_fe['watch_sessions_per_week'] / self.session_p95_, 0, 1)
        completion_norm = np.clip(X_fe['completion_rate'] / 100, 0, 1)
        reco_norm = np.clip(X_fe['recommendation_click_rate'] / 100, 0, 1)
        recency_norm = 1 - np.clip(X_fe['days_since_last_login'] / self.inactive_p95_, 0, 1)
        X_fe['engagement_score'] = 100 * (
            0.25 * watch_norm
            + 0.20 * session_norm
            + 0.25 * completion_norm
            + 0.15 * reco_norm
            + 0.15 * recency_norm
        )

        # Small number of risk flags. 너무 많은 flag는 tree model에서 noise가 될 수 있어 제한한다.
        X_fe['low_watch_flag'] = (X_fe['avg_watch_time_minutes_per_week'] <= self.watch_low_q_).astype(int)
        X_fe['low_session_flag'] = (X_fe['watch_sessions_per_week'] <= self.session_low_q_).astype(int)
        X_fe['low_completion_flag'] = (X_fe['completion_rate'] <= self.completion_low_q_).astype(int)
        X_fe['low_reco_flag'] = (X_fe['recommendation_click_rate'] <= self.reco_low_q_).astype(int)
        X_fe['high_inactivity_flag'] = (X_fe['days_since_last_login'] >= self.inactive_high_q_).astype(int)
        X_fe['inactive_low_watch_flag'] = (
            (X_fe['high_inactivity_flag'] == 1)
            & (X_fe['low_watch_flag'] == 1)
        ).astype(int)
        X_fe['low_interest_flag'] = (
            (X_fe['low_reco_flag'] == 1)
            & (X_fe['low_completion_flag'] == 1)
        ).astype(int)
        X_fe['basic_mobile_flag'] = (
            (X_fe['subscription_type'].astype(str) == 'Basic')
            & (X_fe['primary_device'].astype(str) == 'Mobile')
        ).astype(int)

        risk_cols = [
            'low_watch_flag',
            'low_session_flag',
            'low_completion_flag',
            'low_reco_flag',
            'high_inactivity_flag',
            'inactive_low_watch_flag',
            'low_interest_flag',
            'basic_mobile_flag',
        ]
        X_fe['high_risk_behavior_count'] = X_fe[risk_cols].sum(axis=1)
        return X_fe


# feature set schema 확인
compact_debugger = CompactChurnFeatureEngineer()
X_train_compact = compact_debugger.fit_transform(X_train)
added_cols = sorted(set(X_train_compact.columns) - set(X_train.columns))
print('Original feature count:', X_train.shape[1])
print('Compact FE feature count:', X_train_compact.shape[1])
print('Added feature count:', len(added_cols))
added_cols


Original feature count: 18
Compact FE feature count: 34
Added feature count: 16


['basic_mobile_flag',
 'completion_per_watch_time',
 'engagement_score',
 'high_inactivity_flag',
 'high_risk_behavior_count',
 'inactive_low_watch_flag',
 'inactive_ratio_to_account_age',
 'low_completion_flag',
 'low_interest_flag',
 'low_reco_flag',
 'low_session_flag',
 'low_watch_flag',
 'recommendation_completion_gap',
 'session_per_account_age',
 'watch_time_per_account_age',
 'watch_time_per_session']

## 4. Preprocessor / Metric Helper

AUC는 threshold와 무관하지만 F1은 threshold에 따라 크게 달라진다. 그래서 모델 학습 후 valid set에서 F1이 가장 높은 threshold를 찾는다.


In [4]:
def build_preprocessor(X_schema):
    numeric_features = X_schema.select_dtypes(include='number').columns.tolist()
    categorical_features = X_schema.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    return ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features),
            ('cat', make_ohe(), categorical_features),
        ],
        remainder='drop',
        verbose_feature_names_out=False,
    )


def build_pipeline(X_train_schema, estimator, feature_set='original'):
    steps = []
    if feature_set == 'compact_fe':
        fe = CompactChurnFeatureEngineer()
        X_schema = fe.fit_transform(X_train_schema)
        steps.append(('feature_engineering', fe))
    elif feature_set == 'original':
        X_schema = X_train_schema.copy()
    else:
        raise ValueError(f'Unknown feature_set: {feature_set}')

    steps.extend([
        ('preprocess', build_preprocessor(X_schema)),
        ('model', clone(estimator)),
    ])
    return Pipeline(steps)


def get_scores(model, X_eval):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X_eval)[:, 1]
    raw_scores = model.decision_function(X_eval)
    return (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min())


def tune_threshold_for_f1(y_true, y_score, thresholds=np.arange(0.05, 0.951, 0.005)):
    rows = []
    for threshold in thresholds:
        y_pred = (y_score >= threshold).astype(int)
        rows.append({
            'threshold': threshold,
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
        })
    threshold_df = pd.DataFrame(rows)
    best_row = threshold_df.loc[threshold_df['f1'].idxmax()]
    return best_row, threshold_df


def evaluate_auc_f1(y_true, y_score, threshold=0.5):
    y_pred = (y_score >= threshold).astype(int)
    return {
        'roc_auc': roc_auc_score(y_true, y_score),
        'pr_auc': average_precision_score(y_true, y_score),
        'threshold': threshold,
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'confusion_matrix': confusion_matrix(y_true, y_pred),
    }


def row_without_cm(metrics):
    return {k: v for k, v in metrics.items() if k != 'confusion_matrix'}


## 5. 모델 후보와 Search Space

AUC/F1을 올리는 것이 목적이므로 tree boosting 계열을 우선한다. `N_ITER`를 늘리면 더 오래 걸리지만 성능 탐색 폭이 넓어진다.


In [5]:
def available_model_spaces(y_train):
    spaces = {}

    spaces['LogisticRegression'] = {
        'estimator': LogisticRegression(max_iter=3000, random_state=RANDOM_STATE),
        'params': {
            'model__C': loguniform(0.01, 20),
            'model__class_weight': [None, 'balanced'],
        },
        'n_iter': 12,
    }

    spaces['RandomForest'] = {
        'estimator': RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS),
        'params': {
            'model__n_estimators': randint(300, 800),
            'model__max_depth': [None, 8, 12, 16, 24],
            'model__min_samples_leaf': randint(2, 20),
            'model__max_features': ['sqrt', 'log2', 0.5],
            'model__class_weight': [None, 'balanced', 'balanced_subsample'],
        },
        'n_iter': N_ITER,
    }

    spaces['ExtraTrees'] = {
        'estimator': ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS),
        'params': {
            'model__n_estimators': randint(300, 900),
            'model__max_depth': [None, 8, 12, 16, 24],
            'model__min_samples_leaf': randint(2, 20),
            'model__max_features': ['sqrt', 'log2', 0.5],
            'model__class_weight': [None, 'balanced', 'balanced_subsample'],
        },
        'n_iter': N_ITER,
    }

    try:
        from lightgbm import LGBMClassifier
        spaces['LightGBM'] = {
            'estimator': LGBMClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1),
            'params': {
                'model__n_estimators': randint(300, 1200),
                'model__learning_rate': loguniform(0.01, 0.10),
                'model__num_leaves': randint(16, 96),
                'model__min_child_samples': randint(10, 120),
                'model__subsample': uniform(0.65, 0.35),
                'model__colsample_bytree': uniform(0.65, 0.35),
                'model__reg_alpha': loguniform(1e-4, 10),
                'model__reg_lambda': loguniform(1e-4, 10),
                'model__class_weight': [None, 'balanced'],
            },
            'n_iter': N_ITER,
        }
    except Exception as exc:
        print('[skip] LightGBM unavailable:', exc)

    try:
        from xgboost import XGBClassifier
        neg, pos = np.bincount(y_train)
        spaces['XGBoost'] = {
            'estimator': XGBClassifier(
                eval_metric='logloss',
                tree_method='hist',
                random_state=RANDOM_STATE,
                n_jobs=N_JOBS,
            ),
            'params': {
                'model__n_estimators': randint(300, 1000),
                'model__learning_rate': loguniform(0.01, 0.10),
                'model__max_depth': randint(3, 8),
                'model__min_child_weight': randint(1, 12),
                'model__subsample': uniform(0.65, 0.35),
                'model__colsample_bytree': uniform(0.65, 0.35),
                'model__reg_alpha': loguniform(1e-4, 10),
                'model__reg_lambda': loguniform(1e-4, 10),
                'model__scale_pos_weight': [1, neg / pos, np.sqrt(neg / pos)],
            },
            'n_iter': N_ITER,
        }
    except Exception as exc:
        print('[skip] XGBoost unavailable:', exc)

    try:
        from catboost import CatBoostClassifier
        spaces['CatBoost'] = {
            'estimator': CatBoostClassifier(
                loss_function='Logloss',
                eval_metric='AUC',
                random_seed=RANDOM_STATE,
                verbose=False,
                allow_writing_files=False,
            ),
            'params': {
                'model__iterations': randint(300, 900),
                'model__learning_rate': loguniform(0.01, 0.10),
                'model__depth': randint(4, 9),
                'model__l2_leaf_reg': loguniform(1, 20),
                'model__auto_class_weights': [None, 'Balanced', 'SqrtBalanced'],
            },
            'n_iter': N_ITER,
        }
    except Exception as exc:
        print('[skip] CatBoost unavailable:', exc)

    return spaces


model_spaces = available_model_spaces(y_train)
print('Model candidates:', list(model_spaces.keys()))


Model candidates: ['LogisticRegression', 'RandomForest', 'ExtraTrees', 'LightGBM', 'XGBoost', 'CatBoost']


## 6. Hyperparameter Search

각 모델과 feature set 조합을 CV ROC AUC 기준으로 탐색한다. 이후 valid set에서 F1 threshold를 따로 튜닝한다.

정렬 기준은 `valid_selection_score = valid_roc_auc + valid_best_f1`이다. AUC와 F1을 둘 다 높이는 후보를 위로 올리기 위한 단순한 선택 점수다.


In [6]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
feature_sets = ['original', 'compact_fe']

search_rows = []
fitted_candidates = {}
valid_scores = {}
valid_threshold_tables = {}

for feature_set in feature_sets:
    for model_name, spec in model_spaces.items():
        run_name = f'{model_name}__{feature_set}'
        print(f'[search] {run_name}', flush=True)

        pipe = build_pipeline(X_train, spec['estimator'], feature_set=feature_set)
        search = RandomizedSearchCV(
            estimator=pipe,
            param_distributions=spec['params'],
            n_iter=spec['n_iter'],
            scoring='roc_auc',
            cv=cv,
            n_jobs=N_JOBS,
            random_state=RANDOM_STATE,
            refit=True,
            verbose=1,
        )
        search.fit(X_train, y_train)

        best_model = search.best_estimator_
        y_valid_score = get_scores(best_model, X_valid)
        valid_base = evaluate_auc_f1(y_valid, y_valid_score, threshold=0.5)
        best_threshold_row, threshold_df = tune_threshold_for_f1(y_valid, y_valid_score)
        valid_tuned = evaluate_auc_f1(
            y_valid,
            y_valid_score,
            threshold=float(best_threshold_row['threshold']),
        )

        fitted_candidates[run_name] = best_model
        valid_scores[run_name] = y_valid_score
        valid_threshold_tables[run_name] = threshold_df

        search_rows.append({
            'run_name': run_name,
            'model': model_name,
            'feature_set': feature_set,
            'best_cv_roc_auc': search.best_score_,
            'valid_roc_auc': valid_base['roc_auc'],
            'valid_pr_auc': valid_base['pr_auc'],
            'valid_f1_at_0_5': valid_base['f1'],
            'best_valid_threshold': float(best_threshold_row['threshold']),
            'valid_best_f1': valid_tuned['f1'],
            'valid_precision_at_best_f1': valid_tuned['precision'],
            'valid_recall_at_best_f1': valid_tuned['recall'],
            'valid_selection_score': valid_base['roc_auc'] + valid_tuned['f1'],
            'best_params': search.best_params_,
        })
        print(
            f"[done] {run_name} | "
            f"CV ROC AUC={search.best_score_:.4f}, "
            f"Valid ROC AUC={valid_base['roc_auc']:.4f}, "
            f"Valid best F1={valid_tuned['f1']:.4f}, "
            f"threshold={float(best_threshold_row['threshold']):.3f}",
            flush=True,
        )

search_result = pd.DataFrame(search_rows).sort_values(
    ['valid_selection_score', 'valid_roc_auc', 'valid_best_f1'],
    ascending=False,
).reset_index(drop=True)

search_result[[
    'run_name',
    'best_cv_roc_auc',
    'valid_roc_auc',
    'valid_pr_auc',
    'valid_f1_at_0_5',
    'best_valid_threshold',
    'valid_best_f1',
    'valid_precision_at_best_f1',
    'valid_recall_at_best_f1',
    'valid_selection_score',
]]


[search] LogisticRegression__original
Fitting 3 folds for each of 12 candidates, totalling 36 fits
[done] LogisticRegression__original | CV ROC AUC=0.9101, Valid ROC AUC=0.9166, Valid best F1=0.7107, threshold=0.640
[search] RandomForest__original
Fitting 3 folds for each of 15 candidates, totalling 45 fits


/Users/j3s30p/playdata/프로젝트/SKN_30_project2/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/j3s30p/playdata/프로젝트/SKN_30_project2/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/j3s30p/playdata/프로젝트/SKN_30_project2/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib wor

[done] RandomForest__original | CV ROC AUC=0.9008, Valid ROC AUC=0.9104, Valid best F1=0.6927, threshold=0.510
[search] ExtraTrees__original
Fitting 3 folds for each of 15 candidates, totalling 45 fits
[done] ExtraTrees__original | CV ROC AUC=0.9012, Valid ROC AUC=0.9113, Valid best F1=0.6964, threshold=0.580
[search] LightGBM__original
Fitting 3 folds for each of 15 candidates, totalling 45 fits
[done] LightGBM__original | CV ROC AUC=0.9074, Valid ROC AUC=0.9140, Valid best F1=0.7013, threshold=0.300
[search] XGBoost__original
Fitting 3 folds for each of 15 candidates, totalling 45 fits
[done] XGBoost__original | CV ROC AUC=0.9083, Valid ROC AUC=0.9152, Valid best F1=0.7056, threshold=0.455
[search] CatBoost__original
Fitting 3 folds for each of 15 candidates, totalling 45 fits
[done] CatBoost__original | CV ROC AUC=0.9089, Valid ROC AUC=0.9166, Valid best F1=0.7049, threshold=0.655
[search] LogisticRegression__compact_fe
Fitting 3 folds for each of 12 candidates, totalling 36 fits
[d

/Users/j3s30p/playdata/프로젝트/SKN_30_project2/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/j3s30p/playdata/프로젝트/SKN_30_project2/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/j3s30p/playdata/프로젝트/SKN_30_project2/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib wor

[done] RandomForest__compact_fe | CV ROC AUC=0.8963, Valid ROC AUC=0.9045, Valid best F1=0.6902, threshold=0.330
[search] ExtraTrees__compact_fe
Fitting 3 folds for each of 15 candidates, totalling 45 fits


/Users/j3s30p/playdata/프로젝트/SKN_30_project2/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/j3s30p/playdata/프로젝트/SKN_30_project2/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/j3s30p/playdata/프로젝트/SKN_30_project2/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib wor

[done] ExtraTrees__compact_fe | CV ROC AUC=0.8967, Valid ROC AUC=0.9077, Valid best F1=0.6865, threshold=0.550
[search] LightGBM__compact_fe
Fitting 3 folds for each of 15 candidates, totalling 45 fits
[done] LightGBM__compact_fe | CV ROC AUC=0.9071, Valid ROC AUC=0.9134, Valid best F1=0.7037, threshold=0.340
[search] XGBoost__compact_fe
Fitting 3 folds for each of 15 candidates, totalling 45 fits
[done] XGBoost__compact_fe | CV ROC AUC=0.9080, Valid ROC AUC=0.9147, Valid best F1=0.7050, threshold=0.490
[search] CatBoost__compact_fe
Fitting 3 folds for each of 15 candidates, totalling 45 fits
[done] CatBoost__compact_fe | CV ROC AUC=0.9086, Valid ROC AUC=0.9147, Valid best F1=0.7020, threshold=0.430


,run_name,best_cv_roc_auc,valid_roc_auc,valid_pr_auc,valid_f1_at_0_5,best_valid_threshold,valid_best_f1,valid_precision_at_best_f1,valid_recall_at_best_f1,valid_selection_score
0,LogisticRegression__compact_fe,0.909879,0.916313,0.766096,0.667366,0.320,0.711249,0.662541,0.767686,1.627562
1,LogisticRegression__original,0.910084,0.916554,0.766255,0.677432,0.640,0.710736,0.663079,0.765774,1.627291
2,CatBoost__original,0.908879,0.916620,0.768024,0.686069,0.655,0.704918,0.673043,0.739962,1.621539
3,XGBoost__original,0.908328,0.915168,0.763252,0.702506,0.455,0.705575,0.648000,0.774379,1.620743
4,XGBoost__compact_fe,0.907961,0.914668,0.763633,0.702899,0.490,0.704984,0.664691,0.750478,1.619652
5,LightGBM__compact_fe,0.907111,0.913367,0.759747,0.668766,0.340,0.703654,0.666097,0.745698,1.617020
6,CatBoost__compact_fe,0.908562,0.914694,0.762952,0.696168,0.430,0.702009,0.634957,0.784895,1.616704
7,LightGBM__original,0.907418,0.913980,0.759361,0.667018,0.300,0.701310,0.645498,0.767686,1.615290
8,ExtraTrees__original,0.901187,0.911292,0.752366,0.677831,0.580,0.696364,0.663778,0.732314,1.607655
9,RandomForest__original,0.900840,0.910399,0.743314,0.691485,0.510,0.692674,0.633624,0.763862,1.603073


## 7. Top 후보 Test 평가

최종 test set은 여기서 처음 사용한다. threshold는 valid set에서 고른 값을 그대로 사용한다.


In [7]:
top_k = min(8, len(search_result))
test_rows = []
test_confusions = {}
test_scores = {}

for _, candidate in search_result.head(top_k).iterrows():
    run_name = candidate['run_name']
    model = fitted_candidates[run_name]
    threshold = float(candidate['best_valid_threshold'])

    y_test_score = get_scores(model, X_test)
    base_metrics = evaluate_auc_f1(y_test, y_test_score, threshold=0.5)
    tuned_metrics = evaluate_auc_f1(y_test, y_test_score, threshold=threshold)

    test_scores[run_name] = y_test_score
    test_confusions[run_name] = tuned_metrics['confusion_matrix']

    test_rows.append({
        'run_name': run_name,
        'model': candidate['model'],
        'feature_set': candidate['feature_set'],
        'valid_roc_auc': candidate['valid_roc_auc'],
        'valid_best_f1': candidate['valid_best_f1'],
        'test_roc_auc': base_metrics['roc_auc'],
        'test_pr_auc': base_metrics['pr_auc'],
        'test_f1_at_0_5': base_metrics['f1'],
        'valid_selected_threshold': threshold,
        'test_f1_at_valid_threshold': tuned_metrics['f1'],
        'test_precision_at_valid_threshold': tuned_metrics['precision'],
        'test_recall_at_valid_threshold': tuned_metrics['recall'],
    })

test_result = pd.DataFrame(test_rows).sort_values(
    ['test_roc_auc', 'test_f1_at_valid_threshold'],
    ascending=False,
).reset_index(drop=True)

test_result


,run_name,model,feature_set,valid_roc_auc,valid_best_f1,test_roc_auc,test_pr_auc,test_f1_at_0_5,valid_selected_threshold,test_f1_at_valid_threshold,test_precision_at_valid_threshold,test_recall_at_valid_threshold
0,LogisticRegression__original,LogisticRegression,original,0.916554,0.710736,0.902585,0.752671,0.668740,0.640,0.682184,0.641414,0.728489
1,LogisticRegression__compact_fe,LogisticRegression,compact_fe,0.916313,0.711249,0.902402,0.752660,0.653304,0.320,0.681798,0.637802,0.732314
2,XGBoost__original,XGBoost,original,0.915168,0.705575,0.901569,0.755429,0.676539,0.455,0.683164,0.635168,0.739006
3,CatBoost__original,CatBoost,original,0.916620,0.704918,0.901518,0.754112,0.657844,0.655,0.684626,0.665763,0.704589
4,CatBoost__compact_fe,CatBoost,compact_fe,0.914694,0.702009,0.901310,0.754221,0.686916,0.430,0.685565,0.630313,0.751434
5,XGBoost__compact_fe,XGBoost,compact_fe,0.914668,0.704984,0.900565,0.753772,0.675939,0.490,0.678604,0.652827,0.706501
6,LightGBM__original,LightGBM,original,0.913980,0.701310,0.899821,0.753663,0.662696,0.300,0.676007,0.623586,0.738050
7,LightGBM__compact_fe,LightGBM,compact_fe,0.913367,0.703654,0.899757,0.752189,0.657711,0.340,0.679709,0.648438,0.714149


## 8. Soft Average Ensemble

AUC와 F1을 더 올리기 위해 valid 성능 상위 모델들의 probability를 평균낸다. 복잡한 stacking보다 먼저 soft averaging을 보는 이유는 과적합 위험이 낮고 실험 비용이 작기 때문이다.


In [8]:
ensemble_rows = []
ensemble_scores = {}
ensemble_confusions = {}

# valid 기준 상위 후보를 2~5개까지 평균한다.
ranked_run_names = search_result['run_name'].tolist()
for n_members in range(2, min(5, len(ranked_run_names)) + 1):
    member_names = ranked_run_names[:n_members]

    valid_score_matrix = np.column_stack([valid_scores[name] for name in member_names])
    y_valid_ensemble = valid_score_matrix.mean(axis=1)
    best_threshold_row, _ = tune_threshold_for_f1(y_valid, y_valid_ensemble)
    valid_metrics = evaluate_auc_f1(
        y_valid,
        y_valid_ensemble,
        threshold=float(best_threshold_row['threshold']),
    )

    test_score_matrix = np.column_stack([
        get_scores(fitted_candidates[name], X_test)
        for name in member_names
    ])
    y_test_ensemble = test_score_matrix.mean(axis=1)
    test_base = evaluate_auc_f1(y_test, y_test_ensemble, threshold=0.5)
    test_tuned = evaluate_auc_f1(
        y_test,
        y_test_ensemble,
        threshold=float(best_threshold_row['threshold']),
    )

    ensemble_name = f'SoftAverageTop{n_members}'
    ensemble_scores[ensemble_name] = y_test_ensemble
    ensemble_confusions[ensemble_name] = test_tuned['confusion_matrix']

    ensemble_rows.append({
        'run_name': ensemble_name,
        'members': ', '.join(member_names),
        'valid_roc_auc': valid_metrics['roc_auc'],
        'valid_best_f1': valid_metrics['f1'],
        'valid_selected_threshold': float(best_threshold_row['threshold']),
        'test_roc_auc': test_base['roc_auc'],
        'test_pr_auc': test_base['pr_auc'],
        'test_f1_at_0_5': test_base['f1'],
        'test_f1_at_valid_threshold': test_tuned['f1'],
        'test_precision_at_valid_threshold': test_tuned['precision'],
        'test_recall_at_valid_threshold': test_tuned['recall'],
    })

ensemble_result = pd.DataFrame(ensemble_rows).sort_values(
    ['test_roc_auc', 'test_f1_at_valid_threshold'],
    ascending=False,
).reset_index(drop=True)

ensemble_result


,run_name,members,valid_roc_auc,valid_best_f1,valid_selected_threshold,test_roc_auc,test_pr_auc,test_f1_at_0_5,test_f1_at_valid_threshold,test_precision_at_valid_threshold,test_recall_at_valid_threshold
0,SoftAverageTop4,"LogisticRegression__compact_fe, LogisticRegres...",0.916874,0.707372,0.535,0.902721,0.755451,0.679595,0.681943,0.654930,0.711281
1,SoftAverageTop2,"LogisticRegression__compact_fe, LogisticRegres...",0.916487,0.712135,0.480,0.902610,0.752639,0.678899,0.683014,0.639933,0.732314
2,SoftAverageTop3,"LogisticRegression__compact_fe, LogisticRegres...",0.916898,0.708696,0.510,0.902560,0.753695,0.682231,0.680331,0.625501,0.745698
3,SoftAverageTop5,"LogisticRegression__compact_fe, LogisticRegres...",0.916699,0.708516,0.500,0.902551,0.755952,0.680927,0.680927,0.637730,0.730402


## 9. 최종 후보 선택

최종 선택은 test 결과를 보고 기록한다. 실험 목적이 AUC와 F1 개선이므로, `test_roc_auc`와 `test_f1_at_valid_threshold`를 함께 본다.


In [9]:
model_leaderboard = test_result[[
    'run_name',
    'test_roc_auc',
    'test_pr_auc',
    'test_f1_at_0_5',
    'valid_selected_threshold',
    'test_f1_at_valid_threshold',
    'test_precision_at_valid_threshold',
    'test_recall_at_valid_threshold',
]].copy()

ensemble_leaderboard = ensemble_result[[
    'run_name',
    'test_roc_auc',
    'test_pr_auc',
    'test_f1_at_0_5',
    'valid_selected_threshold',
    'test_f1_at_valid_threshold',
    'test_precision_at_valid_threshold',
    'test_recall_at_valid_threshold',
]].copy()

final_leaderboard = pd.concat([model_leaderboard, ensemble_leaderboard], ignore_index=True)
final_leaderboard = final_leaderboard.sort_values(
    ['test_roc_auc', 'test_f1_at_valid_threshold'],
    ascending=False,
).reset_index(drop=True)

best_auc_row = final_leaderboard.loc[final_leaderboard['test_roc_auc'].idxmax()]
best_f1_row = final_leaderboard.loc[final_leaderboard['test_f1_at_valid_threshold'].idxmax()]

print('Best ROC AUC candidate')
display(best_auc_row.to_frame().T)
print('Best F1 candidate')
display(best_f1_row.to_frame().T)

final_leaderboard


Best ROC AUC candidate


,run_name,test_roc_auc,test_pr_auc,test_f1_at_0_5,valid_selected_threshold,test_f1_at_valid_threshold,test_precision_at_valid_threshold,test_recall_at_valid_threshold
0,SoftAverageTop4,0.902721,0.755451,0.679595,0.535,0.681943,0.65493,0.711281


Best F1 candidate


,run_name,test_roc_auc,test_pr_auc,test_f1_at_0_5,valid_selected_threshold,test_f1_at_valid_threshold,test_precision_at_valid_threshold,test_recall_at_valid_threshold
8,CatBoost__compact_fe,0.90131,0.754221,0.686916,0.43,0.685565,0.630313,0.751434


,run_name,test_roc_auc,test_pr_auc,test_f1_at_0_5,valid_selected_threshold,test_f1_at_valid_threshold,test_precision_at_valid_threshold,test_recall_at_valid_threshold
0,SoftAverageTop4,0.902721,0.755451,0.679595,0.535,0.681943,0.654930,0.711281
1,SoftAverageTop2,0.902610,0.752639,0.678899,0.480,0.683014,0.639933,0.732314
2,LogisticRegression__original,0.902585,0.752671,0.668740,0.640,0.682184,0.641414,0.728489
3,SoftAverageTop3,0.902560,0.753695,0.682231,0.510,0.680331,0.625501,0.745698
4,SoftAverageTop5,0.902551,0.755952,0.680927,0.500,0.680927,0.637730,0.730402
5,LogisticRegression__compact_fe,0.902402,0.752660,0.653304,0.320,0.681798,0.637802,0.732314
6,XGBoost__original,0.901569,0.755429,0.676539,0.455,0.683164,0.635168,0.739006
7,CatBoost__original,0.901518,0.754112,0.657844,0.655,0.684626,0.665763,0.704589
8,CatBoost__compact_fe,0.901310,0.754221,0.686916,0.430,0.685565,0.630313,0.751434
9,XGBoost__compact_fe,0.900565,0.753772,0.675939,0.490,0.678604,0.652827,0.706501


In [10]:
# Confusion matrix 확인: F1 최상 후보 기준
best_f1_name = best_f1_row['run_name']

if best_f1_name in test_confusions:
    best_cm = test_confusions[best_f1_name]
elif best_f1_name in ensemble_confusions:
    best_cm = ensemble_confusions[best_f1_name]
else:
    raise KeyError(best_f1_name)

print('Best F1 candidate:', best_f1_name)
print(best_cm)


Best F1 candidate: CatBoost__compact_fe
[[3493  461]
 [ 260  786]]


## 10. 실험 결과 해석

이번 `07-3` 실험은 `07-1`, `07-2`와 달리 **AUC와 F1 score를 높이는 것**을 최우선 목표로 두었다. 그래서 최종 해석도 캠페인 top-k 지표가 아니라 `test_roc_auc`, `test_pr_auc`, `test_f1_at_valid_threshold` 중심으로 본다.

전체 결과에서 가장 중요한 결론은 다음과 같다.

- **ROC AUC 1위:** `SoftAverageTop4`
- **F1 1위:** `CatBoost__compact_fe`
- **원본 feature set의 강함:** `LogisticRegression__original`이 단일 모델 중 가장 높은 ROC AUC를 기록했다.
- **compact feature의 역할:** 전체적으로 AUC를 크게 끌어올리지는 못했지만, `CatBoost__compact_fe`에서는 F1 개선에 가장 좋은 결과를 만들었다.

즉 AUC 최적화와 F1 최적화의 최종 후보가 다르다. 운영 목표가 ranking 품질이면 `SoftAverageTop4`, threshold 기반 churn 분류 성능이면 `CatBoost__compact_fe`를 우선 검토하는 구조가 적절하다.


## 11. AUC 관점 해석

AUC 기준으로는 ensemble이 가장 좋았다.

`SoftAverageTop4`는 test ROC AUC 0.9027로 최상위 결과를 기록했다. 단일 모델 중에서는 `LogisticRegression__original`이 test ROC AUC 0.9026으로 매우 근접했다. 두 결과 차이는 약 0.0001 수준이라 실질적으로는 거의 같은 성능으로 볼 수 있다.

중요한 점은 `compact_fe`가 AUC를 일관되게 개선하지는 못했다는 것이다. 예를 들어 `LogisticRegression__compact_fe`는 valid에서는 가장 높은 selection score를 보였지만, test ROC AUC에서는 `LogisticRegression__original`보다 소폭 낮았다. XGBoost, LightGBM도 original이 compact_fe보다 대체로 안정적이었다.

따라서 AUC만 기준으로 보면 다음 결론이 자연스럽다.

- 최종 AUC 후보는 `SoftAverageTop4` 또는 `LogisticRegression__original`이다.
- ensemble은 아주 작은 AUC 이득을 만들었지만, 단일 Logistic Regression 대비 복잡도가 높다.
- 성능 차이가 작기 때문에 해석성과 운영 단순성을 중시하면 `LogisticRegression__original`도 충분히 경쟁력 있다.


## 12. F1 관점 해석

F1 기준 최상 후보는 `CatBoost__compact_fe`다.

`CatBoost__compact_fe`는 valid에서 선택한 threshold 0.430을 test에 그대로 적용했을 때 test F1 0.6856을 기록했다. 이 값은 leaderboard에서 가장 높다. confusion matrix는 다음과 같다.

```text
[[3493  461]
 [ 260  786]]
```

이를 해석하면 다음과 같다.

- 실제 non-churn 3,954명 중 3,493명을 non-churn으로 맞췄고, 461명은 churn으로 잘못 예측했다.
- 실제 churn 1,046명 중 786명을 churn으로 잡았고, 260명은 놓쳤다.
- recall은 약 0.7514로, churn 고객을 비교적 많이 포착했다.
- precision은 약 0.6303으로, churn이라고 예측한 고객 중 약 63%가 실제 churn이었다.

즉 `CatBoost__compact_fe`는 AUC 1위는 아니지만, threshold를 적용한 실제 0/1 분류에서는 가장 균형이 좋았다. 이번 목표가 “F1 score를 높이자”라면 이 모델이 가장 직접적인 후보이다.


## 13. Feature Engineering 효과 판단

`compact_fe`는 `07-2`처럼 많은 feature를 추가하지 않고 16개만 추가했다. 주요 피처는 `watch_time_per_session`, `engagement_score`, `inactive_low_watch_flag`, `low_interest_flag`, `basic_mobile_flag`, `high_risk_behavior_count` 등이다.

실험 결과를 보면 compact feature의 효과는 모델별로 달랐다.

- Logistic Regression: original이 compact_fe보다 test ROC AUC가 약간 높고, F1도 거의 비슷하다.
- XGBoost / LightGBM: original이 compact_fe보다 AUC 측면에서 더 안정적이다.
- CatBoost: compact_fe가 F1 기준 최상 결과를 만들었다.
- RandomForest / ExtraTrees: compact_fe가 성능 개선으로 이어지지 않았다.

따라서 compact feature가 모든 모델에서 좋은 것은 아니다. 하지만 CatBoost처럼 feature interaction을 잘 잡는 모델에서는 F1 개선에 도움이 됐다. 이 결과는 `07-2`의 결론과도 연결된다. 많은 파생변수를 무조건 넣는 것은 위험하지만, 행동 신호를 압축한 소수 feature는 특정 모델에서 유효할 수 있다.

최종적으로는 다음처럼 판단할 수 있다.

- **AUC 최우선:** `SoftAverageTop4` 또는 `LogisticRegression__original`
- **F1 최우선:** `CatBoost__compact_fe`
- **단순성과 해석성 우선:** `LogisticRegression__original`
- **성능만 우선:** AUC 후보와 F1 후보를 모두 보관하고 운영 목적에 따라 선택


## 14. 07-3 최종 결론

`07-3`의 목적은 AUC와 F1 score 개선이었다. 실험 결과, 두 지표의 최적 후보가 완전히 같지는 않았다.

AUC 기준으로는 `SoftAverageTop4`가 test ROC AUC 0.9027로 가장 높았다. 다만 `LogisticRegression__original`도 0.9026으로 거의 동일하다. 따라서 AUC만 보면 ensemble의 이득은 매우 작다.

F1 기준으로는 `CatBoost__compact_fe`가 test F1 0.6856으로 가장 높았다. 특히 recall이 0.7514로 높아 churn 고객을 더 많이 잡는 방향의 모델이다. F1을 높이는 것이 이번 실험의 핵심 목표 중 하나였기 때문에, threshold 기반 churn 분류 모델로는 `CatBoost__compact_fe`가 가장 좋은 후보이다.

정리하면 다음과 같다.

1. **risk ranking이 중요하면** `SoftAverageTop4`를 선택한다.
2. **F1 score가 중요하면** `CatBoost__compact_fe`를 선택한다.
3. **단순한 운영과 설명 가능성이 중요하면** `LogisticRegression__original`을 선택한다.
4. `compact_fe`는 전체 모델의 AUC를 일관되게 올리지는 못했지만, CatBoost의 F1 개선에는 도움이 됐다.

따라서 `07-3`의 결론은 “feature engineering 자체가 압도적으로 성능을 올렸다”가 아니라, **compact feature와 threshold tuning을 결합했을 때 F1 기준으로 더 나은 후보를 찾았다**로 정리하는 것이 맞다.


## 15. 해석 기준

이 노트북에서는 다른 지표보다 AUC와 F1을 우선한다.

**AUC가 높아졌다는 의미**

- churn 고객과 non-churn 고객의 risk score 순서가 더 잘 분리됐다는 뜻이다.
- threshold를 아직 정하지 않은 상태에서 모델 ranking 품질이 좋아졌다고 볼 수 있다.
- ROC AUC가 비슷하면 PR AUC를 보조 기준으로 확인한다.

**F1이 높아졌다는 의미**

- precision과 recall의 균형이 좋아졌다는 뜻이다.
- 이 프로젝트에서는 valid set에서 선택한 threshold를 test set에 그대로 적용한 `test_f1_at_valid_threshold`를 가장 중요하게 본다.
- `test_f1_at_0_5`보다 `test_f1_at_valid_threshold`가 높으면 threshold tuning이 효과가 있었던 것이다.

**최종 선택 원칙**

1. `test_roc_auc`가 가장 높은 후보를 먼저 확인한다.
2. 그 후보의 `test_f1_at_valid_threshold`가 지나치게 낮지 않은지 본다.
3. F1을 최우선으로 운영해야 한다면 `test_f1_at_valid_threshold`가 가장 높은 후보를 선택한다.
4. AUC 1위와 F1 1위가 다르면, 이번 목표에 맞춰 둘 중 하나를 명시적으로 선택한다.
5. feature set은 `compact_fe`가 원본보다 AUC/F1을 둘 다 올릴 때만 채택한다. 하나라도 하락하면 원본 피처를 유지한다.
